# Chapter 12: Chain-of-Thought, Few-Shot, and Meta Language Creation
### Design of Agentic Systems with Case Studies

This notebook demonstrates three prompt architectures for structured data extraction from messy clinical text and deliberately triggers three failure modes.

**Architectures covered:**
- Zero-Shot
- Few-Shot + Chain-of-Thought
- Meta Language DSL (ANALYZE / FORMAT / VERIFY)

**Failure modes triggered:**
- Format Overfitting (star failure)
- Exemplar Bias
- CoT Collapse

In [ ]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.7 MB/s eta 0:00:00


## Setup
Add your Anthropic API key to Colab Secrets:
1. Click the key icon in the left sidebar
2. Add a secret named `ANTHROPIC_API_KEY`
3. Paste your key (starts with sk-ant-...)
4. Toggle "Notebook access" ON
Then run this cell.

In [ ]:
from groq import Groq
from google.colab import userdata

api_key = userdata.get('my-secret-key')
client = Groq(api_key=api_key)

def call_claude(system_prompt, user_prompt, max_tokens=1500):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

print("✅ Setup complete. Groq API connected.")

✅ Setup complete. Groq API connected.


## The Test Input

This is the messy clinical note used throughout the chapter.
It contains 5 medication entries. Not one can be represented
as a confident, fully-populated structured record.

**Correct output should show:**
- Lisinopril → CONFLICTING dose (family says 10mg, rx bottle says 20mg)
- Rosuvastatin → UNCONFIRMED (patient calls it "the little pink one")
- Warfarin → HOLD + ACTION-REQUIRED
- Metformin → PARTIAL-ADHERENCE flag

In [ ]:
MESSY_INPUT = """
pt takes metFORMIN 500 twice daily (says she sometimes skips PM dose),
also on lisinopril — dose unclear, family says 10mg but rx bottle says 20,
Asprin 81mg QD, 'the little pink one for cholesterol' — probably rosuvastatin
per PCP note attached, hold warfarin pre-procedure per attending
"""

print("Test input loaded:")
print(MESSY_INPUT)

Test input loaded:

pt takes metFORMIN 500 twice daily (says she sometimes skips PM dose),
also on lisinopril — dose unclear, family says 10mg but rx bottle says 20,
Asprin 81mg QD, 'the little pink one for cholesterol' — probably rosuvastatin
per PCP note attached, hold warfarin pre-procedure per attending



---
## Architecture 1: Zero-Shot

**What it is:** The model receives only the task description and the input.
No examples, no reasoning steps, no schema.

**Architectural bet:** The model can infer correct behavior from training alone.

**Expected failure:** The model will resolve ambiguity silently —
fabricating dose values and dropping low-salience entries like the warfarin hold.

In [ ]:
ZERO_SHOT_SYSTEM = """
You are a clinical medication extraction system.
Extract all medications from the clinical note provided.
Return a JSON list of medication records with these fields:
drug_name, dose, frequency, route, status, flags.
"""

print("Running Zero-Shot extraction...\n")
zero_shot_output = call_claude(ZERO_SHOT_SYSTEM, MESSY_INPUT)
print(zero_shot_output)

Running Zero-Shot extraction...

Here is the list of extracted medications in JSON format:

```
[
  {
    "drug_name": "Metformin",
    "dose": "500mg",
    "frequency": "twice daily",
    "route": "oral",
    "status": "active",
    "flags": ["non-adherent"]
  },
  {
    "drug_name": "Lisinopril",
    "dose": "10-20mg",
    "frequency": "daily",
    "route": "oral",
    "status": "active",
    "flags": ["dose uncertain"]
  },
  {
    "drug_name": "Aspirin",
    "dose": "81mg",
    "frequency": "daily",
    "route": "oral",
    "status": "active",
    "flags": []
  },
  {
    "drug_name": "Rosuvastatin",
    "dose": "unknown",
    "frequency": "daily",
    "route": "oral",
    "status": "active",
    "flags": ["probable", "dose unknown"]
  },
  {
    "drug_name": "Warfarin",
    "dose": "unknown",
    "frequency": "unknown",
    "route": "oral",
    "status": "held",
    "flags": ["held pre-procedure"]
  }
]
```


### Zero-Shot — What to Observe

Look for these signals in the output above:

1. **Lisinopril dose** — is it 10mg, 20mg, or shown as a conflict?
2. **Rosuvastatin dose** — is there a fabricated numeric value?
3. **Warfarin** — did the hold instruction survive?
4. **Flags** — are any fields marked as uncertain or unconfirmed?

Zero-shot does not fail loudly. It fails quietly, with confident, well-formatted JSON.

---
## Architecture 2: Few-Shot + Chain-of-Thought

**What it is:** The model receives worked examples (exemplars) and an instruction
to reason through each medication before producing structured output.

**Architectural mechanism:** The reasoning step acts as externalized working memory —
it forces the model to enumerate every entry before committing to output tokens.
Because every output token is conditioned on preceding tokens, a reasoning step
that explicitly names an entry makes it improbable for the output to omit it.

**Improvement over zero-shot:** Ambiguity is surfaced rather than silently resolved.

In [ ]:
FEW_SHOT_SYSTEM = """
You are a clinical medication extraction system.
Before producing structured output, reason through each medication
mentioned in the note, flag ambiguities, then produce the structured list.

EXAMPLE INPUT:
"Patient on metformin 500mg BID, also takes something for blood pressure
— doesn't know the name, lisinopril per old records but not confirmed"

EXAMPLE OUTPUT:
REASONING:
- metformin 500mg BID: confirmed, dose and frequency explicit
- blood pressure medication: patient self-report only, name unconfirmed,
  old records suggest lisinopril but not verified in this encounter

RECORDS:
[
  {"drug_name": "metformin", "dose": "500mg", "frequency": "BID",
   "status": "ACTIVE", "flags": []},
  {"drug_name": "UNCONFIRMED (possibly lisinopril)", "dose": null,
   "frequency": null, "status": "ACTIVE",
   "flags": ["UNCONFIRMED", "requires verification"]}
]
"""

print("Running Few-Shot + CoT extraction...\n")
few_shot_output = call_claude(FEW_SHOT_SYSTEM, MESSY_INPUT)
print(few_shot_output)

Running Few-Shot + CoT extraction...

REASONING:
- metformin 500mg twice daily: confirmed, dose and frequency explicit, but patient admits to sometimes skipping the PM dose, indicating non-adherence.
- lisinopril: confirmed, but dose ambiguous due to discrepancy between family report (10mg) and rx bottle (20mg), requires clarification.
- Aspirin 81mg QD: confirmed, dose and frequency explicit.
- 'the little pink one for cholesterol': likely rosuvastatin based on PCP note, but not explicitly confirmed by the patient, requires verification.
- warfarin: confirmed to be held pre-procedure, but no information on typical dose or frequency.

RECORDS:
[
  {"drug_name": "metformin", "dose": "500mg", "frequency": "twice daily",
   "status": "ACTIVE", "flags": ["NON-ADHERENT"]},
  {"drug_name": "lisinopril", "dose": "UNCLEAR (10mg or 20mg)",
   "frequency": null, "status": "ACTIVE", "flags": ["DOSE UNCLEAR", "requires verification"]},
  {"drug_name": "Aspirin", "dose": "81mg", "frequency": "QD",


### Few-Shot + CoT — What to Observe

Compare this output to the zero-shot output above:

1. **Reasoning trace** — did the model enumerate all 5 entries before formatting?
2. **Lisinopril** — is the conflict between 10mg and 20mg now visible?
3. **Rosuvastatin** — is it flagged as UNCONFIRMED rather than given a fabricated dose?
4. **Warfarin** — did the hold instruction survive this time?

The reasoning step is not decorative. It is working memory.
It prevented the model from compressing the input before processing all of it.

---
## Architecture 3: Meta Language DSL — Baseline (Correct Version)

**What it is:** A formally defined Domain-Specific Language embedded in the prompt.
Three phases: ANALYZE (reasoning trace) → FORMAT (structured output) → VERIFY (consistency check).

**Key design decision:** UNCONFIRMED and CONFLICTING are first-class tokens in the schema.
This means the model can satisfy a required field with an honest uncertainty representation
rather than a fabricated value.

**Save this output** — you will compare it to the modified version in the failure case below.

In [ ]:
META_LANGUAGE_BASELINE = """
You are a clinical medication extraction system using a structured
Meta Language protocol. Process every note in three phases:

PHASE 1 — ANALYZE:
Walk through the note linearly. For every medication-adjacent statement,
record: what was said, the source, and your epistemic assessment.
Do not skip any entry. Do not produce structured output yet.

PHASE 2 — FORMAT:
For each entry in ANALYZE, produce a structured record using ONLY this schema.
Draw values ONLY from your ANALYZE findings.

Schema:
  drug_name: [string | UNCONFIRMED]
  dose: [numeric+unit | CONFLICTING(source_a, source_b) | UNCONFIRMED | null]
  frequency: [QD | BID | TID | PRN | UNCONFIRMED | null]
  route: [PO | IV | SQ | UNCONFIRMED | null]
  status: [ACTIVE | HOLD | DISCONTINUED | UNCONFIRMED]
  flags: [CONFLICTING | UNCONFIRMED | PARTIAL-ADHERENCE | ACTION-REQUIRED | INFERRED]
  source: [quote the exact text this record came from]

PHASE 3 — VERIFY:
Count entries in ANALYZE vs FORMAT. Confirm every flag from ANALYZE
appears in FORMAT. Report:
  entries_identified_in_analyze: [integer]
  entries_in_format: [integer]
  unresolved_flags: [list]
  reviewer_action_required: [YES | NO]
  reviewer_action_reason: [string | null]
"""

print("Running Meta Language DSL baseline...\n")
baseline_output = call_claude(META_LANGUAGE_BASELINE, MESSY_INPUT, max_tokens=2000)
print(baseline_output)

Running Meta Language DSL baseline...

## PHASE 1 — ANALYZE

1. The patient takes metFORMIN 500 twice daily.
   - What was said: The patient takes metFORMIN 500 twice daily.
   - Source: Patient statement.
   - Epistemic assessment: The dose and frequency are confirmed, but there's a mention of sometimes skipping the PM dose, which may indicate partial adherence.

2. The patient is on lisinopril, with a dose unclear between 10mg (family's statement) and 20mg (rx bottle).
   - What was said: Dose of lisinopril is unclear.
   - Source: Family statement and rx bottle.
   - Epistemic assessment: There is a conflict in the reported dose of lisinopril.

3. The patient takes Asprin 81mg QD.
   - What was said: The patient takes Asprin 81mg QD.
   - Source: Direct statement.
   - Epistemic assessment: The dose and frequency of Asprin are confirmed.

4. The patient takes 'the little pink one for cholesterol', likely rosuvastatin.
   - What was said: The patient takes 'the little pink one for ch

### Meta Language DSL Baseline — What to Observe

This is the correct output. Verify all of these are present:

-  Lisinopril: `dose: CONFLICTING(10mg family report, 20mg rx bottle)`
-  Rosuvastatin: `dose: UNCONFIRMED`
-  Warfarin: `status: HOLD`, `flags: [ACTION-REQUIRED]`
-  Metformin: `flags: [PARTIAL-ADHERENCE]`
-  VERIFY: `reviewer_action_required: YES`

Save or screenshot this output before running the next cell.

---
## MANDATORY HUMAN DECISION NODE

The Meta Language schema above assumes that UNCONFIRMED and CONFLICTING
are first-class tokens in the dose field vocabulary.

**This condition does NOT hold in many real-world deployments** where:
- Downstream pharmacy systems require a numeric dose to trigger drug-interaction checks
- EHR write-back modules reject records with non-numeric dose fields
- Legacy systems were built assuming extraction always produces confident values

**Before proceeding, document your decision:**

Q1: Does your downstream system accept null or UNCONFIRMED in required fields?
Q2: If not, where in the pipeline does human review occur?
Q3: Is silent failure (fabricated dose passes validation) acceptable in your domain?

> **My decision:** I am keeping UNCONFIRMED and CONFLICTING as first-class tokens
> because the clinical cost of a silent failure (fabricated dose acted on by a pharmacist)
> outweighs the engineering cost of redesigning the downstream verification system.
> The alternative — accepting silent failure — is not architecturally acceptable in a
> high-stakes clinical domain.

---
##  FAILURE MODE 1: Format Overfitting

**The modification:** One line change — remove UNCONFIRMED and CONFLICTING
from the dose field's valid tokens. The schema now requires a numeric value only.

**Prediction — answer these before running:**
1. What dose value will appear for lisinopril (conflict: 10mg vs 20mg)?
2. What dose value will appear for rosuvastatin (no dose stated in note)?
3. Will VERIFY catch either failure?

Write your predictions now, before running the cell.

In [ ]:
# ============================================================
# THE ONE LINE CHANGE
# Before: dose: [numeric+unit | CONFLICTING(source_a, source_b) | UNCONFIRMED | null]
# After:  dose: [numeric+unit]
#
# This removes all uncertainty tokens from the dose field.
# The model can no longer represent ambiguity — only fabricate values.
# ============================================================

META_LANGUAGE_OVERFITTING = """
You are a clinical medication extraction system using a structured
Meta Language protocol. Process every note in three phases:

PHASE 1 — ANALYZE:
Walk through the note linearly. For every medication-adjacent statement,
record: what was said, the source, and your epistemic assessment.
Do not skip any entry. Do not produce structured output yet.

PHASE 2 — FORMAT:
For each entry in ANALYZE, produce a structured record using ONLY this schema.

Schema:
  drug_name: [string]
  dose: [numeric+unit]
  frequency: [QD | BID | TID | PRN | null]
  route: [PO | IV | SQ | null]
  status: [ACTIVE | HOLD | DISCONTINUED]
  flags: [PARTIAL-ADHERENCE | ACTION-REQUIRED | INFERRED]
  source: [quote the exact text this record came from]

PHASE 3 — VERIFY:
Count entries in ANALYZE vs FORMAT. Report:
  entries_identified_in_analyze: [integer]
  entries_in_format: [integer]
  reviewer_action_required: [YES | NO]
  reviewer_action_reason: [string | null]
"""

print("Running Format Overfitting failure case...\n")
overfitting_output = call_claude(META_LANGUAGE_OVERFITTING, MESSY_INPUT, max_tokens=2000)
print(overfitting_output)

Running Format Overfitting failure case...

PHASE 1 — ANALYZE:
1. The patient takes metFORMIN 500 twice daily, but sometimes skips the PM dose. 
   Source: The patient's statement.
   Epistemic assessment: The information about metFORMIN is directly from the patient, but there is uncertainty about adherence.

2. The patient is on lisinopril, with a dose unclear between 10mg (as per family) and 20mg (as per the RX bottle).
   Source: Family statement and RX bottle.
   Epistemic assessment: There is discrepancy in the reported dose, requiring further verification.

3. The patient is on Aspirin 81mg QD.
   Source: Patient's statement.
   Epistemic assessment: The information about Aspirin seems clear and direct from the patient.

4. The patient is on 'the little pink one for cholesterol', likely rosuvastatin as per the PCP note.
   Source: Patient's description and PCP note.
   Epistemic assessment: The identity of the medication is inferred from the patient's description and supported by

### Format Overfitting — Signals to Find

**Signal 1 — Lisinopril:**
The dose field now contains a single numeric value instead of CONFLICTING.
Check the ANALYZE trace — the conflict is almost certainly still documented there.
The conflict appears in ANALYZE and disappears in FORMAT.
This is format overfitting in its exact mechanism.

**Signal 2 — Rosuvastatin:**
The dose field contains a numeric value. Search for that value in the input note.
You will not find it. It has no source in the text.
The model inferred it from genre conventions (rosuvastatin starter doses are commonly 10mg).

**Signal 3 — VERIFY:**
Did VERIFY flag the discrepancy between ANALYZE and FORMAT?
Or did it only check that all fields are populated?
These are different checks. The latter is not a safeguard.

**Signal 4 — Source field:**
The rosuvastatin source field correctly quotes "the little pink one for cholesterol."
The source looks right. The dose is fabricated.
The record is internally inconsistent in a way automated validation cannot detect.

**Why this is dangerous:**
Every record validates. No flags indicate a problem.
The JSON is well-formed. The downstream system accepts it.
The failure is completely invisible.

---
## FAILURE MODE 2: Exemplar Bias

**The setup:** Exemplar set drawn only from clean inputs with compliant patients.
Model encounters a NOAC (direct oral anticoagulant) entry and misclassifies
the monitoring note as a non-adherence signal.

**The causal chain:**
One exemplar established: monitoring language = PARTIAL-ADHERENCE flag.
The model did not learn this from the exemplar alone — it activated a prior
from training (monitoring language is associated with non-adherence across
millions of clinical documents). The exemplar confirmed that prior.
One exemplar was enough because the training distribution had already primed
the association.

In [ ]:
OCR_INPUT = """
Pt on metfornin 500mg BID (diabetcs), lisinopri 1 Omg QD,
atorvasttin 40mg QD nightly, baby asprin daily,
Xare1to 20mg w/ evening meal — INR not monitored (NOAC)
"""

EXEMPLAR_BIAS_SYSTEM = """
You are a clinical medication extraction system.
Before producing structured output, reason through each medication,
then produce the structured list.

EXAMPLE INPUT:
"Patient takes warfarin 5mg daily — INR last checked 3 weeks ago,
patient admits sometimes forgetting doses"

EXAMPLE OUTPUT:
REASONING:
- warfarin 5mg QD: confirmed
- INR monitoring gap noted — patient non-adherent to monitoring protocol

RECORDS:
[{"drug_name": "warfarin", "dose": "5mg", "frequency": "QD",
  "status": "ACTIVE", "flags": ["PARTIAL-ADHERENCE"]}]
"""

print("Running Exemplar Bias failure case...\n")
exemplar_bias_output = call_claude(EXEMPLAR_BIAS_SYSTEM, OCR_INPUT)
print(exemplar_bias_output)

Running Exemplar Bias failure case...

REASONING:
- metformin 500mg BID: confirmed, used for diabetes management
- lisinopril 10mg QD: confirmed, assumed to be for hypertension or other cardiovascular indication
- atorvastatin 40mg QD: confirmed, used for cholesterol management
- baby aspirin (81mg) daily: confirmed, used for antiplatelet therapy
- Xarelto 20mg QD with evening meal: confirmed, used for anticoagulation; note that as a NOAC, INR monitoring is not required, but it's essential to assess for signs of bleeding or thrombosis

RECORDS:
[
  {"drug_name": "metformin", "dose": "500mg", "frequency": "BID", "status": "ACTIVE", "flags": []},
  {"drug_name": "lisinopril", "dose": "10mg", "frequency": "QD", "status": "ACTIVE", "flags": []},
  {"drug_name": "atorvastatin", "dose": "40mg", "frequency": "QD", "status": "ACTIVE", "flags": []},
  {"drug_name": "aspirin", "dose": "81mg", "frequency": "QD", "status": "ACTIVE", "flags": []},
  {"drug_name": "Xarelto", "dose": "20mg", "frequen

### Exemplar Bias — What to Observe

1. **Rivaroxaban (Xarelto) flag** — is PARTIAL-ADHERENCE applied?
2. **Why this is wrong** — INR monitoring is not required for NOACs.
   Rivaroxaban is a direct oral anticoagulant. It does not require INR monitoring.
   "INR not monitored" is a clinical clarification, not a compliance gap.
3. **The fix** — add a counterexample demonstrating a NOAC entry with
   an INR-not-required note, correctly producing no adherence flag,
   placed immediately before the new input in the exemplar sequence.

The model generalized correctly from the evidence it was given.
The failure is in the evidence.

---
## FAILURE MODE 3: CoT Collapse

**The setup:** A corrupted reasoning step in the CoT exemplar encodes a
precedence rule — "outside records take precedence over patient self-report."
This rule is clinically reasonable as a default.
In the new input, the outside records themselves are under active pharmacy dispute.
The exemplar's rule does not account for this case.

**The danger:** The wrong reasoning step is fully documented in the ANALYZE trace.
A human reviewer who reads it will find the error immediately.
But in a production pipeline where FORMAT is consumed programmatically
and ANALYZE is discarded, the reasoning trace is not a safeguard.
It is an archaeological record of a failure that was already operationalized.

In [ ]:
WARFARIN_INPUT = """
Patient reports taking Coumadin, says dose is 5mg but last INR per outside
records was 3.8 on 2.5mg — dose discrepancy flagged by pharmacy.
Holding warfarin per hematology pending repeat INR tomorrow.
Also on levothyroxine 88mcg QD, last TSH 1.2 (within range).
Metformin 1000mg BID, patient reports compliance.
"""

# THE CORRUPTED EXEMPLAR
# The reasoning rule is correct in the exemplar's context.
# It is wrong when the outside records are themselves disputed.
COT_COLLAPSE_SYSTEM = """
You are a clinical medication extraction system.
Before producing structured output, reason through each medication,
then produce the structured list.

EXAMPLE INPUT:
"Patient on warfarin, reports 5mg daily. Outside hospital records show
last stable dose was 3mg. Hold per cardiology pre-procedure."

EXAMPLE OUTPUT:
ANALYZE: Warfarin entry. Hold instruction present. When a hold instruction
is present, status is HOLD and dose is recorded as last confirmed dose.
Outside records are authoritative over patient self-report when a discrepancy
exists. Last confirmed dose from outside records: 3mg.

RECORDS:
[{"drug_name": "warfarin", "dose": "3mg", "frequency": "QD",
  "status": "HOLD", "flags": ["ACTION-REQUIRED"]}]
"""

print("Running CoT Collapse failure case...\n")
cot_collapse_output = call_claude(COT_COLLAPSE_SYSTEM, WARFARIN_INPUT)
print(cot_collapse_output)

Running CoT Collapse failure case...

ANALYZE: Coumadin (warfarin) entry. Dose discrepancy present between patient report (5mg) and outside records (2.5mg). Hold instruction present due to pending repeat INR. When a hold instruction is present, status is HOLD and dose is recorded as last confirmed dose from outside records: 2.5mg.

ANALYZE: Levothyroxine entry. No dose discrepancy or hold instruction present. Patient's last TSH level is within range, indicating stable dosing.

ANALYZE: Metformin entry. No dose discrepancy or hold instruction present. Patient reports compliance, but this is a self-reported measure and may require further verification.

RECORDS:
[
  {"drug_name": "warfarin", "dose": "2.5mg", "frequency": "QD", 
   "status": "HOLD", "flags": ["DOSE-DISCREPANCY", "ACTION-REQUIRED"]},
  {"drug_name": "levothyroxine", "dose": "88mcg", "frequency": "QD", 
   "status": "ACTIVE", "flags": []},
  {"drug_name": "metformin", "dose": "1000mg", "frequency": "BID", 
   "status": "ACT

### CoT Collapse — What to Observe

1. **Warfarin dose** — is it shown as 2.5mg (outside records) or CONFLICTING?
2. **ANALYZE trace** — does the model document the precedence rule it applied?
3. **FORMAT output** — does it look correct? (HOLD status, ACTION-REQUIRED flag)
4. **The gap** — the error is visible in ANALYZE but invisible in FORMAT.

A downstream pharmacist reading only FORMAT sees:
- A valid dose value (2.5mg)
- A HOLD status (correct)
- An ACTION-REQUIRED flag (looks like it's working)

Nothing in FORMAT signals that the dose was resolved by a rule
that should not have applied here.

**The fix:** Qualify the precedence rule in the exemplar:
"Outside records take precedence UNLESS they are themselves the subject
of an active discrepancy flag." Add this as a counterexample.

---
## Summary: Three Architectures, Three Failure Modes

| Architecture | Failure Mode | Why It Fails | Output Signal |
|---|---|---|---|
| Zero-Shot | Silent confabulation | No mechanism to represent uncertainty | None — looks correct |
| Few-Shot + CoT | Exemplar bias | Exemplar set misses input type | Wrong flag — detectable |
| Meta Language DSL | Format overfitting | Schema removes uncertainty paths | None — validates cleanly |

---

## The Design Rule

Before deploying any prompt architecture, run it against the hardest 20% of
your real inputs. Examine not whether it produces output — but what that output
looks like when it is wrong.

If wrong outputs are indistinguishable from right outputs,
the architecture is not ready for deployment.

---

## The Open Question

The VERIFY phase is generated by the same model that generated FORMAT.
A model that resolved a conflict silently in FORMAT may evaluate its own
output as consistent in VERIFY — because the same reasoning that produced
the silent resolution is still in its context window.

One candidate fix: pass only the ANALYZE trace and FORMAT output to VERIFY,
explicitly excluding the task description and exemplar set.
This removes the reasoning environment that produced the FORMAT output
from the verification context.

Whether this fully escapes the problem — or merely displaces it —
is what this chapter leaves open.